# Module 16: ML Workflow End-to-End

**Lesson: Project Structure, Pipelines, MLflow, Evaluation, and Interpretation**

This notebook covers the complete end-to-end ML project lifecycle with modern practices.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_california_housing, load_iris
from sklearn.model_selection import (
    train_test_split, cross_val_score, learning_curve, validation_curve
)
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.metrics import (
    mean_squared_error, r2_score, confusion_matrix, ConfusionMatrixDisplay,
    classification_report
)
from sklearn.inspection import permutation_importance
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid')
print('Libraries loaded successfully')

## 1. ML Project Structure

A well-organized ML project separates concerns: data loading, feature engineering, model training, evaluation, and configuration.

In [ ]:
# Example project structure (for reference, not executed)
project_structure = '''
ml_project/
    configs/
        config.yaml
    data/
        raw/
        processed/
    notebooks/
        01_eda.ipynb
        02_modeling.ipynb
    src/
        data/
            loader.py
        features/
            build_features.py
        models/
            train_model.py
            predict.py
    models/
        model.joblib
    reports/
        figures/
    mlruns/         # MLflow artifacts
    requirements.txt
    README.md
'''
print('Standard ML project structure:')
print(project_structure)

## 2. Pipeline Design with ColumnTransformer

In [ ]:
# Load California Housing
housing = fetch_california_housing(as_frame=True)
df = housing.frame

X = df.drop('MedHouseVal', axis=1)
y = df['MedHouseVal']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Build a pipeline with ColumnTransformer
numeric_features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']

numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features)
    ]
)

# Full pipeline with regressor
pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(n_estimators=100, random_state=42))
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print('=== Pipeline Performance ===')
print(f'R2: {r2_score(y_test, y_pred):.4f}')
print(f'RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')

## 3. Model Evaluation — Learning Curves and Validation Curves

In [ ]:
# Learning curve: diagnose bias vs variance
train_sizes, train_scores, val_scores = learning_curve(
    pipeline, X_train, y_train, cv=5, n_jobs=-1,
    train_sizes=np.linspace(0.1, 1.0, 10), scoring='r2'
)

train_mean = np.mean(train_scores, axis=1)
train_std = np.std(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)
val_std = np.std(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(train_sizes, train_mean, 'o-', label='Training score', color='blue')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.2, color='blue')
plt.plot(train_sizes, val_mean, 'o-', label='Cross-validation score', color='red')
plt.fill_between(train_sizes, val_mean - val_std, val_mean + val_std, alpha=0.2, color='red')
plt.xlabel('Training examples')
plt.ylabel('R2 score')
plt.title('Learning Curve: Random Forest on California Housing')
plt.legend(loc='best')
plt.grid(True)
plt.show()
print('Small gap between train and CV scores suggests low variance.')

In [ ]:
# Validation curve: tune n_estimators
param_range = [10, 50, 100, 200, 300]
train_scores, val_scores = validation_curve(
    pipeline, X_train, y_train,
    param_name='regressor__n_estimators',
    param_range=param_range, cv=5, scoring='r2', n_jobs=-1
)

train_mean = np.mean(train_scores, axis=1)
val_mean = np.mean(val_scores, axis=1)

plt.figure(figsize=(8, 5))
plt.plot(param_range, train_mean, 'o-', label='Training', color='blue')
plt.plot(param_range, val_mean, 'o-', label='Cross-validation', color='red')
plt.xlabel('n_estimators')
plt.ylabel('R2 score')
plt.title('Validation Curve: n_estimators')
plt.legend()
plt.grid(True)
plt.show()
print(f'Best n_estimators: {param_range[np.argmax(val_mean)]}')

## 4. Confusion Matrix and Classification Report

In [ ]:
# Classification example on Titanic with pipeline
titanic = sns.load_dataset('titanic')
titanic = titanic[['survived', 'pclass', 'sex', 'age', 'fare', 'embarked']].copy()

X_t = titanic.drop('survived', axis=1)
y_t = titanic['survived']

X_train_t, X_test_t, y_train_t, y_test_t = train_test_split(
    X_t, y_t, test_size=0.2, random_state=42, stratify=y_t
)

num_cols = ['age', 'fare']
cat_cols = ['pclass', 'sex', 'embarked']

clf_pipeline = Pipeline([
    ('preprocessor', ColumnTransformer([
        ('num', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler())
        ]), num_cols),
        ('cat', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
        ]), cat_cols)
    ])),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

clf_pipeline.fit(X_train_t, y_train_t)
y_pred_t = clf_pipeline.predict(X_test_t)

print('=== Classification Report ===')
print(classification_report(y_test_t, y_pred_t, target_names=['Died', 'Survived']))

ConfusionMatrixDisplay.from_predictions(y_test_t, y_pred_t, display_labels=['Died', 'Survived'])
plt.title('Confusion Matrix')
plt.show()

## 5. Model Interpretation — Feature Importance and SHAP

In [ ]:
# Permutation importance
perm_importance = permutation_importance(
    pipeline, X_test, y_test, n_repeats=10, random_state=42, n_jobs=-1
)

feature_names = X.columns
importance_df = pd.DataFrame({
    'feature': feature_names,
    'importance': perm_importance.importances_mean,
    'std': perm_importance.importances_std
}).sort_values('importance', ascending=False)

print('=== Permutation Feature Importance ===')
print(importance_df)

plt.figure(figsize=(8, 4))
sns.barplot(data=importance_df, x='importance', y='feature', xerr=importance_df['std'])
plt.title('Feature Importance (Permutation)')
plt.tight_layout()
plt.show()

In [ ]:
# SHAP basics for model interpretation
try:
    import shap
    
    # Use a smaller sample for speed
    X_sample = X_test[:100]
    
    # Get the preprocessed data
    X_processed = pipeline[:-1].transform(X_sample)
    
    # Create SHAP explainer for the Random Forest
    rf_model = pipeline[-1]
    explainer = shap.TreeExplainer(rf_model)
    shap_values = explainer.shap_values(X_processed)
    
    # Summary plot
    plt.figure()
    shap.summary_plot(shap_values, X_processed, feature_names=feature_names, show=False)
    plt.title('SHAP Feature Importance')
    plt.tight_layout()
    plt.show()
    
except ImportError:
    print('SHAP not installed. Install with: pip install shap')

## 6. Experiment Tracking with MLflow

In [ ]:
try:
    import mlflow
    from mlflow.models import infer_signature
    
    with mlflow.start_run(run_name='california_housing_rf') as run:
        # Log parameters
        mlflow.log_param('model_type', 'RandomForestRegressor')
        mlflow.log_param('n_estimators', 100)
        mlflow.log_param('max_depth', None)
        
        # Log metrics
        mlflow.log_metric('r2', r2_score(y_test, y_pred))
        mlflow.log_metric('rmse', np.sqrt(mean_squared_error(y_test, y_pred)))
        
        # Log the model
        signature = infer_signature(X_train, y_train)
        mlflow.sklearn.log_model(pipeline, 'model', signature=signature)
        
        # Log artifacts
        plt.figure(figsize=(8, 4))
        sns.barplot(data=importance_df.head(8), x='importance', y='feature')
        plt.title('Feature Importance')
        plt.tight_layout()
        plt.savefig('feature_importance.png', dpi=100, bbox_inches='tight')
        mlflow.log_artifact('feature_importance.png')
        plt.close()
        
        print(f'MLflow run ID: {run.info.run_id}')
        print(f'Logged R2: {r2_score(y_test, y_pred):.4f}')
        print(f'Logged RMSE: {np.sqrt(mean_squared_error(y_test, y_pred)):.4f}')
        
except ImportError:
    print('MLflow not installed. Install with: pip install mlflow')

## 7. Model Serialization

In [ ]:
import joblib
import os

# Save the full pipeline
model_path = 'california_housing_pipeline.joblib'
joblib.dump(pipeline, model_path)
print(f'Model saved to {model_path}')
print(f'File size: {os.path.getsize(model_path) / 1024 / 1024:.2f} MB')

# Load and verify
loaded_pipeline = joblib.load(model_path)
y_pred_loaded = loaded_pipeline.predict(X_test[:5])
print('\nLoaded model predictions (first 5):')
print(y_pred_loaded)
print('Original predictions (first 5):')
print(y_pred[:5])
print('\nPredictions match:', np.allclose(y_pred_loaded, y_pred[:5]))

## Summary

- **Project structure**: Separate data, features, models, configs for reproducibility
- **Pipelines**: sklearn Pipeline + ColumnTransformer = no data leakage, easy deployment
- **Learning curves**: Diagnose bias (high gap) vs variance (high CV gap)
- **Validation curves**: Find optimal hyperparameter values
- **Model interpretation**: Permutation importance (global), SHAP (local + global)
- **Experiment tracking**: MLflow logs params, metrics, artifacts, models
- **Serialization**: joblib for sklearn models, include full pipeline
- **End-to-end**: Raw data → Pipeline → Evaluation → Interpretation → Save → Deploy